## Setting up data

In [34]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('..'))  # go up one level from matching_model to code

### Preparing visitor data

In [ ]:
from data_cleaning.parse_visitors import parse_visitor_data

# load raw csvs
visitors = pd.read_csv("../../source/visitors.csv")
answers = pd.read_csv("../../source/visitors_answers.csv")
questions = pd.read_csv("../../source/visitors_questions.csv")

In [4]:
# parse
visitor_data = parse_visitor_data(visitors, answers, questions)
visitor_data.head()

,email,gender,visitorId,stepId,questionId,answerValue,answerId,answerTypeId,answer,questionTypeId,question
14,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f730fd,5c8a78336d41a10da4f730fe,,5c8a78336d41a10da4f73100,Answer,To obtain general information,5bf7c399b82beb7a182cc3de,Reason for Attending the Event
20,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f73225,5c8a78336d41a10da4f73227,,5c8a78336d41a10da4f73244,Answer,Media,5bf7c399b82beb7a182cc3de,Which of the following best describes your job...
23,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f73252,5c8a78336d41a10da4f73253,,5c8a78336d41a10da4f73291,Answer,Travel Agent,5bf7c399b82beb7a182cc3de,Please indicate your company's main area of bu...
31,emilija+100_L8gA@bss.mk,F,67b70a9f2d21f543a1096602,5c8a78336d41a10da4f7336c,5c8a78336d41a10da4f7336d,,5c8a78336d41a10da4f73371,Answer,No influence,5bf7c399b82beb7a182cc3de,What role do you play in the purchasing decisi...
47,aleksandar.dimkov+mitt1_n5eA@bss.com.mk,M,67ada1ee197e604dd2722d1b,5c8a78336d41a10da4f730fd,5c8a78336d41a10da4f730fe,,5c8a78336d41a10da4f730ff,Answer,To source products and services,5bf7c399b82beb7a182cc3de,Reason for Attending the Event


In [5]:
from data_cleaning.visitor_interest_mapping import map_visitor_interests

visitor_interested_categories = map_visitor_interests(visitor_data)
visitor_interested_categories.head()

,visitorId,email,interest_category_list
0,0wcaegyyobblhhvfzwyibn0a,daniela.p+203_NfBj_lrIk@bss.com.mk,{Tour Operators}
1,1o5g2tlsho4gxzk3usr06z1c,sergey.usenko_ucwx_DPxA@ite.group,{}
2,35d97gdotgwa5lwpd0vs7k0h,tanja+182_jiPa_5IoL@bss.com.mk,{Travel Agencies}
3,3a17inawhgac2o0xtix3fb3n,aleksandar.dimkov+mitt1_n5eA_oG2a@bss.com.mk,{MICE}
4,3p80z1iocd67z0qvg8ju1cc0,aleksandar.dimkov+mitt10_V0iB_2bw2@bss.com.mk,{Tour Operators}


### Preparing exhibitor data

In [35]:
from data_cleaning.parse_exhibitors import parse_exhibitor_data

# load raw csvs
exhibitors = pd.read_csv("../../source/exhibitors.csv")
categories = pd.read_csv("../../source/exhibitor_categories.csv")

In [36]:
exhibitor_data = parse_exhibitor_data(exhibitors, categories)
exhibitor_data.head()

,exhibitorId,Name,categoryId,categoryName
0,90556,Turkey Travels,52276,1.5 Resort hotel
1,90556,Turkey Travels,52280,2.1 Inbound tour operator
2,90556,Turkey Travels,52281,2.2 Outbound tour operator
3,92462,Russian Travel Company,52273,1.2 Apartments / Residential hotel
4,92462,Russian Travel Company,52283,2.4 Mass market tour operators


In [44]:
from data_cleaning.exhibitor_category_mapping import map_exhibitor_category

exhibitor_data = map_exhibitor_category(exhibitor_data)
exhibitor_data.head()

,exhibitorId,Name,categoryId,category,category_class_name
0,90556,Turkey Travels,52276,Resort hotel,Hotels & Stays
1,90556,Turkey Travels,52280,Inbound tour operator,Tour Operators
2,90556,Turkey Travels,52281,Outbound tour operator,Tour Operators
3,92462,Russian Travel Company,52273,Apartments / Residential hotel,Hotels & Stays
4,92462,Russian Travel Company,52283,Mass market tour operators,Tour Operators


## Mapping exhibitors to visitors

### Data recap

Lets now look at the clean visitor and exhibitor data

In [45]:
visitor_interested_categories.head()

,visitorId,email,interest_category_list
0,0wcaegyyobblhhvfzwyibn0a,daniela.p+203_NfBj_lrIk@bss.com.mk,{Tour Operators}
1,1o5g2tlsho4gxzk3usr06z1c,sergey.usenko_ucwx_DPxA@ite.group,{}
2,35d97gdotgwa5lwpd0vs7k0h,tanja+182_jiPa_5IoL@bss.com.mk,{Travel Agencies}
3,3a17inawhgac2o0xtix3fb3n,aleksandar.dimkov+mitt1_n5eA_oG2a@bss.com.mk,{MICE}
4,3p80z1iocd67z0qvg8ju1cc0,aleksandar.dimkov+mitt10_V0iB_2bw2@bss.com.mk,{Tour Operators}


In [47]:
exhibitor_data.head()

,exhibitorId,Name,categoryId,category,category_class_name
0,90556,Turkey Travels,52276,Resort hotel,Hotels & Stays
1,90556,Turkey Travels,52280,Inbound tour operator,Tour Operators
2,90556,Turkey Travels,52281,Outbound tour operator,Tour Operators
3,92462,Russian Travel Company,52273,Apartments / Residential hotel,Hotels & Stays
4,92462,Russian Travel Company,52283,Mass market tour operators,Tour Operators


We see that the category class names were engineered to match with the category class names in the interest category list. We can now optimize the matches by penalizing spammy exhibitors (having multiple categories)

In [68]:
# Preparing a dictionary with exhibitor names as keys and their list of category classes as values
exhibitor_dict = exhibitor_data.groupby('Name')['category_class_name'].apply(lambda x: set(x)).to_dict()

In [69]:
# Preparing a dictionary with visitor email ids as keys and their list of interest categories as values
visitor_dict = visitor_interested_categories.set_index('email')['interest_category_list'].to_dict()

### Writing util functions to compute the matching score for exhibitors

In [70]:
def penalized_score(visitor_set, exhibitor_set):
    """
    Computes a penalized Jaccard similarity between two sets. Here, Jaccard similarity computes the similarity between two sets of strings.
    We want to penalize exhibitors who list too many categories, so we reduce their score separately. Lowest score is 0, if either of the sets are empty.
    """
    # reusing our category threshold from our exhibitor_profile_analysis.ipynb, where we declare the exhibitor_class to be spammy, if its higher than this number.
    category_threshold = 4

    # if either set is empty, nothing to score
    if not visitor_set or not exhibitor_set: 
        return 0.0

    intersection = visitor_set & exhibitor_set
    union = visitor_set | exhibitor_set
    jaccard = len(intersection) / len(union)

    # only penalize if exhibitor has more categories than category threshold
    if len(exhibitor_set) > category_threshold:
        penalty = 1/len(exhibitor_set)
    else:
        penalty = 1

    return jaccard * penalty


In [71]:
def recommend_exhibitors(visitor_email, top_k=7):
    """
    Returns top_k matching exhibitors for a given visitor.
    Scores are based on category overlap with optional penalization.
    """
    # initializing empty exhibitor score dict    
    exhibitor_scores = dict()

    visitor_interest_set = set(visitor_dict.get(visitor_email, set()))

    for exhibitor_name, exhibitor_cats in exhibitor_dict.items():
        score = penalized_score(visitor_interest_set, exhibitor_cats)
        exhibitor_scores[exhibitor_name] = score

    # we want to rank exhibitors by their score (descending),
    ranked_exhibitors = sorted(exhibitor_scores.items(), key=lambda item: item[1], reverse=True) # sort by score in descending order

    return ranked_exhibitors[:top_k] # if the visitor has expressed no interest, they will receive random recommendations for exhibitors


### Identifying top exhibitors per visitor

In [73]:
# Given the email address of a visitor, we can list down the top matched exhibitors, with their ranked score
recommend_exhibitors("daniela.p+203_NfBj_lrIk@bss.com.mk")

[('Global Holidays Expeditions', 0.5),
 ('Indian Travel Company', 0.5),
 ('Turkey Travels', 0.5),
 ('DreamTravel Company ', 0.3333333333333333),
 ('Exotic Adventures Journeys', 0.3333333333333333),
 ('Exotic Europe Travels', 0.3333333333333333),
 ('Exotic Tours Tours', 0.04000000000000001)]